# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = '/content/drive/My Drive/cs3264/'

In [ ]:
!pip install great_tables

import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import log_loss, accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import lightgbm as lgb
import numpy as np
from plotnine import *
from itertools import combinations
from great_tables import GT

In [ ]:
df = pl.read_csv(drive_path + 'diabetes_prediction_dataset.csv')
synthetic_df = pl.read_csv(drive_path + 'synthetic_diabetes_data_positive.csv')

In [ ]:
df = df.with_columns(
    pl.col('gender').cast(pl.Categorical),
    pl.col('smoking_history').cast(pl.Categorical)
)
synthetic_df = synthetic_df.with_columns(
    pl.col('gender').cast(pl.Categorical),
    pl.col('smoking_history').cast(pl.Categorical)
)

# XGB Implementation

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df.drop('diabetes'), df['diabetes'], test_size=0.2, stratify=df['diabetes'], random_state=42)
dtrain = xgb.DMatrix(x_train, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(x_test, label=y_test, enable_categorical=True)

In [ ]:
clf = xgb.XGBClassifier(n_estimators=1000, enable_categorical=True)
hyperparams = {
  'learning_rate': stats.loguniform(1e-4, 1e-1),
  'max_depth': stats.randint(1, 9),
  'subsample': stats.uniform(0.5, 0.5),
  'reg_lambda': stats.loguniform(1e-4, 1e2),
}
search = RandomizedSearchCV(clf, hyperparams, n_iter=50, scoring='f1')
search.fit(x_train, y_train)

In [ ]:
search_res = pl.DataFrame(search.cv_results_)
search_res.sort('rank_test_score', descending=True).head(20)

In [ ]:
list(search.best_params_.values())[:-1]

In [ ]:
hp_range = pl.DataFrame({
    'Hyperparameter': list(hyperparams.keys()),
    'Distribution': [
        'log-uniform(1e-4, 1e-1)',
        '(discrete) uniform(1, 8)',
        'uniform(0.5, 1)',
        'log-uniform(1e-4, 1e2)',
    ],
    'Best Value': list(search.best_params_.values())
})
print(GT(hp_range).fmt_number(columns='Best Value', decimals=5).as_latex())

In [ ]:
hps = list(map(lambda x: 'param_' + x, list(hyperparams.keys())))
for hp in hps:
  display(
      ggplot(search_res, aes(x=hp, y='mean_test_score')) +
      geom_point()
  )

for x,y in combinations(hps, 2):
  display(
      ggplot(search_res, aes(x=x, y=y, color='mean_test_score')) +
      geom_point()
  )

In [ ]:
params = search.best_params_
params['objective'] = 'binary:logistic'
params

In [ ]:
xgb.cv(params, dtrain, 5000, early_stopping_rounds=100, nfold=10, seed=42)

In [ ]:
evallist = [(dtrain, 'train'), (dtest, 'eval')]
bst = xgb.train(params, dtrain, 1000, evals=evallist)

In [ ]:
bst.save_model(drive_path + 'model.ubj')

In [ ]:
bst = xgb.Booster({'nthread': 4})
bst.load_model(drive_path + 'model.ubj')

In [ ]:
y_pred = bst.predict(xgb.DMatrix(x_test, enable_categorical=True))
metrics = {
    'F1 Score': f1_score(y_test, y_pred.round()),
    'F1 Macro Score': f1_score(y_test, y_pred.round(), average='macro'),
    'Precision': precision_score(y_test, y_pred.round()),
    'Recall': recall_score(y_test, y_pred.round()),
    'Accuracy': accuracy_score(y_test, y_pred.round()),
    'Log Loss': log_loss(y_test, y_pred),
}
metrics_tab = GT(pl.DataFrame({
    'Metric': list(metrics.keys()),
    'Value': list(metrics.values())
})).fmt_number(columns='Value', decimals=5)
metrics_tab

In [ ]:
print(metrics_tab.as_latex())

# XGB with Augmented Data

In [ ]:
df_aug = pl.concat([df, synthetic_df])
x_train, x_test, y_train, y_test = train_test_split(df_aug.drop('diabetes'), df_aug['diabetes'], test_size=0.2, stratify=df_aug['diabetes'], random_state=42)
dtrain = xgb.DMatrix(x_train, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(x_test, label=y_test, enable_categorical=True)

In [ ]:
clf = xgb.XGBClassifier(n_estimators=1000, enable_categorical=True)
hyperparams = {
  'learning_rate': stats.loguniform(1e-4, 1e-1),
  'max_depth': stats.randint(1, 9),
  'subsample': stats.uniform(0.5, 0.5),
  'reg_lambda': stats.loguniform(1e-4, 1e2),
}
search = RandomizedSearchCV(clf, hyperparams, n_iter=50, scoring='f1')
search.fit(x_train, y_train)

In [ ]:
search_res = pl.DataFrame(search.cv_results_)
search_res.sort('rank_test_score', descending=True).head(20)

In [ ]:
search.best_params_

In [ ]:
hp_range = pl.DataFrame({
    'Hyperparameter': list(hyperparams.keys()),
    'Distribution': [
        'log-uniform(1e-4, 1e-1)',
        '(discrete) uniform(1, 8)',
        'uniform(0.5, 1)',
        'log-uniform(1e-4, 1e2)',
    ],
    'Best Value': list(search.best_params_.values())
})
(GT(hp_range).fmt_number(columns='Best Value', decimals=5))

In [ ]:
hps = list(map(lambda x: 'param_' + x, list(hyperparams.keys())))
for hp in hps:
  display(
      ggplot(search_res, aes(x=hp, y='mean_test_score')) +
      geom_point()
  )

for x,y in combinations(hps, 2):
  display(
      ggplot(search_res, aes(x=x, y=y, color='mean_test_score')) +
      geom_point()
  )

In [ ]:
params = search.best_params_
params['objective'] = 'binary:logistic'
params

In [ ]:
xgb.cv(params, dtrain, 5000, early_stopping_rounds=100, nfold=10, seed=42)

In [ ]:
evallist = [(dtrain, 'train'), (dtest, 'eval')]
bst = xgb.train(params, dtrain, 1500, evals=evallist)

In [ ]:
bst.save_model(drive_path + 'aug_data_model.ubj')

In [ ]:
from joblib import dump
dump(bst, drive_path + 'aug_data_gbt_model.joblib')

In [ ]:
bst = xgb.Booster({'nthread': 4})
bst.load_model(drive_path + 'aug_data_model.ubj')

In [ ]:
y_pred = bst.predict(xgb.DMatrix(x_test, enable_categorical=True))
metrics = {
    'F1 Score': f1_score(y_test, y_pred.round()),
    'F1 Macro Score': f1_score(y_test, y_pred.round(), average='macro'),
    'Precision': precision_score(y_test, y_pred.round()),
    'Recall': recall_score(y_test, y_pred.round()),
    'Accuracy': accuracy_score(y_test, y_pred.round()),
    'Log Loss': log_loss(y_test, y_pred),
}
metrics_tab = GT(pl.DataFrame({
    'Metric': list(metrics.keys()),
    'Value': list(metrics.values())
})).fmt_number(columns='Value', decimals=5)
metrics_tab

In [ ]:
print(metrics_tab.as_latex())

# Feature Engineering Testing

In [ ]:
df_bmi_class = df.with_columns(
    pl.when(pl.col('bmi') < 18.5).then(pl.lit('Underweight'))
    .when((pl.col('bmi') >= 18.5) & (pl.col('bmi') < 25)).then(pl.lit('Normal'))
    .when((pl.col('bmi') >= 25) & (pl.col('bmi') < 30)).then(pl.lit('Overweight'))
    .when((pl.col('bmi') >= 30) & (pl.col('bmi') < 35)).then(pl.lit('Obese 1'))
    .otherwise(pl.lit('Obese 2'))
    .alias('bmi_class')
    .cast(pl.Categorical)
)

In [ ]:
def split_data(df):
  x_train, x_test, y_train, y_test = train_test_split(df.drop('diabetes'), df['diabetes'], test_size=0.2, random_state=42)
  dtrain = xgb.DMatrix(x_train, label=y_train, enable_categorical=True)
  dtest = xgb.DMatrix(x_test, label=y_test, enable_categorical=True)
  evallist = [(dtrain, 'train'), (dtest, 'eval')]
  return x_train, x_test, y_train, y_test, dtrain, dtest, evallist

In [ ]:
x_train, x_test, y_train, y_test, dtrain, dtest, evallist = split_data(df_bmi_class)

In [ ]:
xgb.cv(param, dtrain, 5000, early_stopping_rounds=100, nfold=10, seed=42)

In [ ]:
bst_bmi_binned = xgb.train(param, dtrain, 2500, evals=evallist, verbose_eval=False)

In [ ]:
log_loss(y_test, bst_bmi_binned.predict(xgb.DMatrix(x_test, enable_categorical=True)))

In [ ]:
xgb.plot_importance(bst_bmi_binned)

In [ ]:
df_hba1c_binned = df.with_columns(
    pl.when(pl.col('HbA1c_level') < 5.7).then(pl.lit('Normal'))
    .when((pl.col('HbA1c_level') >= 5.7) & (pl.col('HbA1c_level') < 6.5)).then(pl.lit('Pre-Diabetic'))
    .when(pl.col('HbA1c_level') >= 6.5).then(pl.lit('Diabetic'))
    .alias('HbA1c_bin')
    .cast(pl.Categorical)
)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df_hba1c_binned.drop('diabetes'), df_hba1c_binned['diabetes'], test_size=0.2, random_state=42)
dtrain = xgb.DMatrix(x_train, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(x_test, label=y_test, enable_categorical=True)
param = {
    'objective':'binary:logistic',
    'eta':0.01,
    'subsample':0.8,
    'max_depth':3,
    'lambda':5,
    }
evallist = [(dtrain, 'train'), (dtest, 'eval')]

In [ ]:
bst_hba1c_binned = xgb.train(param, dtrain, 2500, evals=evallist, verbose_eval=False)

In [ ]:
log_loss(y_test, bst_hba1c_binned.predict(xgb.DMatrix(x_test, enable_categorical=True)))

In [ ]:
xgb.plot_importance(bst_hba1c_binned)

In [ ]:
df_bmi_age_int = df.with_columns(
    bmi_age = pl.col('bmi') * pl.col('age')
)

In [ ]:
x_train, x_test, y_train, y_test, dtrain, dtest, evallist = split_data(df_bmi_age_int)
bst_bmi_age = xgb.train(param, dtrain, 2500, evals=evallist, verbose_eval=False)
log_loss(y_test, bst_bmi_age.predict(xgb.DMatrix(x_test, enable_categorical=True)))

In [ ]:
xgb.plot_importance(bst_bmi_age)

In [ ]:
df_interaction_terms = df.with_columns(
    bmi_age = pl.col('bmi') * pl.col('age'),
    HbA1c_glucose = pl.col('HbA1c_level') * pl.col('blood_glucose_level'),
    hypertension_heart = pl.col('hypertension') * pl.col('heart_disease'),
)
int_terms = ['bmi_age', 'HbA1c_glucose', 'hypertension_heart']
res = pl.DataFrame(schema = {
    'terms': pl.String,
    'log_loss': pl.Float64
})
for i in range(1 << len(int_terms)):
  incl = [int_terms[j] for j in range(len(int_terms)) if (i & (1 << j))]
  excl = [int_terms[j] for j in range(len(int_terms)) if (i & (1 << j))==0]
  df_working = df_interaction_terms.drop(excl)
  x_train, x_test, y_train, y_test, dtrain, dtest, evallist = split_data(df_working)
  bst_working = xgb.train(param, dtrain, 2500, evals=evallist, verbose_eval=False)
  new_row = pl.DataFrame({'terms': ', '.join(incl), 'log_loss': log_loss(y_test, bst_working.predict(xgb.DMatrix(x_test, enable_categorical=True)))})
  res = res.extend(new_row)
print(res)

# LightGBM Implementation

In [ ]:
train_data = lgb.Dataset(x_train, label=y_train, categorical_feature=['gender', 'smoking_history'])
test_data = lgb.Dataset(x_test, label=y_test, categorical_feature=['gender', 'smoking_history'])

In [ ]:
lgb_param = {
    'objective': 'binary',
    'force_row_wise':True,
    'eta':0.01,
    'subsample':0.8,
    'max_leaves':5,
    'lambda':5
}

In [ ]:
res = lgb.cv(lgb_param, train_data, num_round, nfold=10)

In [ ]:
lgb_model = lgb.train(lgb_param, train_data, 3000)

In [ ]:
lgb_model.save_model(drive_path + 'lgb_model.txt')

In [ ]:
lgb_model = lgb.Booster(model_file=drive_path + 'lgb_model.txt')

In [ ]:
y_pred = lgb_model.predict(x_test)
accuracy_score(y_test, y_pred.round())

# Feature Importance Comparison

In [ ]:
xgb.plot_importance(bst)

In [ ]:
lgb.plot_importance(lgb_model)

In [ ]:
vif_test = df.drop(['diabetes','smoking_history','gender'])

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(X):
    vif = pd.DataFrame()
    vif["feature"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.to_numpy(), i) for i in range(X.shape[1])]
    return vif.sort_values("VIF", ascending=False)

compute_vif(vif_test)